### IPL Data Analytics - Reusable Analysis Functions


In [1]:
import os
print("Working directory:", os.getcwd())
print("Files here:", os.listdir())

import pandas as pd

from collections import defaultdict 

Working directory: E:\neha work\Data Analytics\Projects\IPL sales analaysis
Files here: ['.ipynb_checkpoints', 'Analysis.ipynb', 'chart.ipynb', 'Clean.ipynb', 'data', 'IPL_Dataset_2008_to_2026.csv']


In [2]:
TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

In [3]:
def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x:sorted(set(x))),
        won_by=('match_won_by_clean', 'first')
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']


    played, wins= defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
          played[t] +=1
        wins[row['won_by']] +=1

    team_stats = pd.DataFrame({
        'team':list(played.keys()),
        'matches_played':[played[t] for t in played],
        'wins': [wins.get(t, 0) for t in played],
        
    })

    team_stats['win_pct'] = (team_stats['wins']/team_stats['matches_played']*100).round(1)
    return team_stats.sort_values('win_pct', ascending=False).reset_index(drop=True)

if __name__ == "__main__":
    #quick manualy check
    df =pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)
    print(compute_team_win_percentage(df))
      

    

                           team  matches_played  wins  win_pct
0        Rising Pune Supergiant              16    15     93.8
1                Gujarat Titans              77    47     61.0
2           Chennai Super Kings             264   148     56.1
3                Mumbai Indians             287   155     54.0
4         Kolkata Knight Riders             271   140     51.7
5   Royal Challengers Bengaluru             280   143     51.1
6              Rajasthan Royals             245   123     50.2
7           Sunrisers Hyderabad             206   102     49.5
8          Lucknow Super Giants              70    34     48.6
9                  Punjab Kings             271   126     46.5
10               Delhi Capitals             272   125     46.0
11                Gujarat Lions              29    13     44.8
12         Kochi Tuskers Kerala              14     6     42.9
13              Deccan Chargers              75    29     38.7
14                Pune Warriors              45    12  